In [22]:
#credcard
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from config import config as Config


def analize_credit_card(card_url):

    credential = AzureKeyCredential(Config.key)

    document_Client =  DocumentIntelligenceClient(Config.endpoint, credential)

    card_info = document_Client.begin_analyze_document(
           "prebuilt_cred_card", AnalyzeDocumentRequest(url_source=card_url))
    result_card_info = card_info.result()


    for document in result_card_info.documents:
        fields = document.get("fields", {})

        return {
            "card_nome": fields.get('CardHoldName', {}).get('content'),
            "card_number":fields.get('CardNumber', {}).get('content'),
            "expiry_date": fields.get('ExpirationDate', {}).get('content'),
            "bank_name": fields.get('IssuingBank', {}).get('content')
        }

In [1]:
#blob
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.identity import AzureKeyCredential
from config import config as Config


def analize_credit_card(card_url):

    credential = AzureKeyCredential(Config.key)

    document_Client =  DocumentIntelligenceClient(Config.endpoint, credential)

    card_info = document_Client.begin_analyze_document(
           "prebuilt_cred_card", AnalyzeDocumentRequest(url_source=card_url))
    result_card_info = card_info.result()


    for document in result_card_info.documents:
        fields = document.get("fields", {})

        return {
            "card_nome": fields.get('CardHoldName', {}).get('content'),
            "card_number":fields.get('CardNumber', {}).get('content'),
            "expiry_date": fields.get('ExpirationDate', {}).get('content'),
            "bank_name": fields.get('IssuingBank', {}).get('content')
        }

ImportError: cannot import name 'AzureKeyCredential' from 'azure.identity' (/usr/local/lib/python3.12/dist-packages/azure/identity/__init__.py)

In [2]:
!pip install azure-storage-blob

In [17]:
from azure.storage.blob import BlobServiceClient
import config
import importlib
import os # Ensure os is imported here as well

# Ensure config module is reloaded to pick up latest environment variables from Colab Secrets
importlib.reload(config)
from config import config as Config

# Debugging: Check values from Config class
print(f"DEBUG: Config.azure_storage_connection_string = {Config.azure_storage_connection_string}")
print(f"DEBUG: Config.conteiner_nome = {Config.conteiner_nome}")

# Caminho para o seu arquivo de imagem local
local_image_path = '/content/card-229830_640.jpg' # << SUBSTITUA POR UM CAMINHO REAL DE IMAGEM
blob_name = 'cartao_para_analise.jpg' # Nome que o arquivo terá no blob storage

# Conecte-se ao Azure Blob Storage
blob_service_client = BlobServiceClient.from_connection_string(Config.azure_storage_connection_string)
container_client = blob_service_client.get_container_client(Config.conteiner_nome)

# Upload do arquivo
print(f"Fazendo upload de {local_image_path} para o blob {blob_name} no contêiner {Config.conteiner_nome}...")
with open(file=local_image_path, mode="rb") as data:
    blob_client = container_client.upload_blob(name=blob_name, data=data, overwrite=True)
print(f"Upload concluído: {blob_name}")

# Obter a URL pública do blob
# Nota: O acesso público deve ser configurado no seu contêiner para que esta URL funcione sem autenticação.
blob_url = f"https://{blob_service_client.account_name}.blob.core.windows.net/{Config.conteiner_nome}/{blob_name}"
print(f"URL do blob: {blob_url}")

SecretNotFoundError: Secret AZURE_DOCUMENT_INTELLIGENCE_KEY does not exist.

Com a imagem carregada para o Azure Blob Storage e sua URL gerada, agora podemos usá-la na função `analize_credit_card`.

In [24]:
# Use a URL do blob para analisar o cartão de crédito
card_details = analize_credit_card(blob_url)
print(card_details)

NameError: name 'blob_url' is not defined

In [19]:
#blob
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from config import config as Config


def analize_credit_card(card_url):

    credential = AzureKeyCredential(Config.key)

    document_Client =  DocumentIntelligenceClient(Config.endpoint, credential)

    card_info = document_Client.begin_analyze_document(
           "prebuilt_cred_card", AnalyzeDocumentRequest(url_source=card_url))
    result_card_info = card_info.result()


    for document in result_card_info.documents:
        fields = document.get("fields", {})

        return {
            "card_nome": fields.get('CardHoldName', {}).get('content'),
            "card_number":fields.get('CardNumber', {}).get('content'),
            "expiry_date": fields.get('ExpirationDate', {}).get('content'),
            "bank_name": fields.get('IssuingBank', {}).get('content')
        }

In [21]:
card_image_url = 'https://pixabay.com/pt/illustrations/ce-mapa-cart%c3%a3o-de-ce-atm-1058106/'
card_details = analize_credit_card(card_image_url)
print(card_details)

HttpResponseError: (InvalidRequest) Invalid request.
Code: InvalidRequest
Message: Invalid request.
Inner error: {
    "code": "InvalidContent",
    "message": "Could not download the file from the given URL."
}

In [14]:
%%writefile config.py
import os
from google.colab import userdata

class config:
    key = userdata.get('AZURE_DOCUMENT_INTELLIGENCE_KEY')
    endpoint = os.getenv('endpoint')
    azure_storage_connection_string = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
    conteiner_nome = userdata.get('AZURE_CONTAINER_NAME')

Overwriting config.py


In [15]:
#app.py
import os
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()

class config:
    key = userdata.get('AZURE_DOCUMENT_INTELLIGENCE_KEY')
    endpoint = os.getenv('endpoint')
    azure_storage_connection_string = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
    conteiner_nome = userdata.get('AZURE_CONTAINER_NAME')

SecretNotFoundError: Secret AZURE_DOCUMENT_INTELLIGENCE_KEY does not exist.

In [9]:
!pip install azure-ai-documentintelligence azure-identity python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.3/191.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.7 MB/s eta 0:00:00


In [16]:
import os

# As chaves sensíveis serão carregadas do Colab Secrets.
# Certifique-se de que 'AZURE_DOCUMENT_INTELLIGENCE_KEY', 'AZURE_STORAGE_CONNECTION_STRING',
# e 'AZURE_CONTAINER_NAME' estão configurados nos Segredos do Colab.

# Configurações do Azure Document Intelligence (apenas endpoint, a chave virá dos Secrets)
os.environ['endpoint'] = 'https://cartaofraudedioeastus01.cognitiveservices.azure.com/'

# As configurações do Azure Storage Blob (connection string e container name) virão dos Secrets
print("Variáveis de ambiente do Azure configuradas (com chaves sensíveis agora nos Segredos do Colab)!")

Variáveis de ambiente do Azure configuradas (com chaves sensíveis agora nos Segredos do Colab)!


Com as variáveis de ambiente configuradas na célula anterior, seus módulos `config.py` e `app.py` já podem acessá-las corretamente.